# 53 — Chunking Strategies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/53-chunking-strategies/chunking_workbook.ipynb)

```
Document
   |
   +-- Fixed-size    -> uniform blocks
   +-- Recursive     -> respects structure
   +-- Sentence-win  -> sentence + neighbors
   +-- Semantic      -> embedding boundaries
        |
     Vectorstore -> Retrieve -> Generate
```

## What you will learn

- How four chunking strategies split the same document differently
- Why chunk boundaries matter for retrieval precision
- How to pick the right strategy based on your content type
- How to measure the practical impact of chunking on RAG answer quality

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain-text-splitters==0.3.11",
        "langchain-openai==0.3.33",
        "langchain-chroma==0.2.6",
        "python-dotenv==1.1.1",
    ], check=True)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

In [ ]:
SAMPLE_TEXT = """Neural networks are computing systems loosely inspired by biological neural networks. Deep learning uses multiple layers to progressively extract higher-level features from raw input.

The transformer architecture introduced in 2017 revolutionized natural language processing. It replaces recurrent networks with self-attention, allowing all tokens to attend to each other simultaneously. This parallelism enables training on much larger datasets.

Retrieval-augmented generation (RAG) combines a language model with an external knowledge base. When a question arrives, a retriever fetches relevant documents, and the generator conditions its output on both the question and the retrieved context. This grounds the model in factual sources.

Vector databases store high-dimensional embeddings and support approximate nearest neighbor search. Systems like ChromaDB, Qdrant, and Pinecone make it practical to search millions of document chunks in milliseconds. The embedding quality is as important as the retrieval algorithm.

Evaluation of RAG pipelines requires specialized metrics. Faithfulness checks whether the answer is grounded in the retrieved context. Context recall checks whether all necessary information was retrieved. Answer relevancy checks whether the answer directly addresses the question."""

print(f"Document: {len(SAMPLE_TEXT)} characters, {len(SAMPLE_TEXT.split())} words")
print(f"Paragraphs: {len([p for p in SAMPLE_TEXT.split(chr(10)*2) if p.strip()])}")